In [3]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
import pandas as pd
import os
import numpy as np

# Danh sách đường dẫn file
log_files = [
    "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/exp/AudioStego/logs_5cases_final/1_NoRandom/log2025-10-10_00-07-55.csv",
    "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/exp/AudioStego/logs_5cases_final/2_Random_Fixed_DefaultSalt/benchmark.csv",
    "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/exp/AudioStego/logs_5cases_final/3_Random_Fixed_ContentSalt/benchmark.csv",
    "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/exp/AudioStego/logs_5cases_final/4_Random_Adaptive_DefaultSalt/benchmark.csv",
    "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/exp/AudioStego/logs_5cases_final/5_Random_Adaptive_ContentSalt/benchmark.csv",
    "/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/exp/logs/benchmark_local_input_7_PhaseCoding_full_20260129_092309.csv"
]

def calculate_time_metric(df):
    # Chuẩn hóa tên cột
    df.columns = df.columns.str.strip().str.lower()
    cols = df.columns.tolist()

    direct_time_cols = [c for c in cols if any(x in c for x in ['duration', 'elapsed', 'process']) and 'time' not in c]
    if not direct_time_cols:
         direct_time_cols = [c for c in cols if 'time' in c and 'stamp' not in c and 'date' not in c]

    if direct_time_cols:
        col_name = direct_time_cols[0]
        return pd.to_numeric(df[col_name], errors='coerce').mean()

    timestamp_cols = [c for c in cols if any(x in c for x in ['timestamp', 'date', 'created'])]

    if timestamp_cols:
        col_name = timestamp_cols[0]
        try:
            times = pd.to_datetime(df[col_name], errors='coerce')
            times = times.dropna().sort_values()

            if len(times) < 2: return np.nan

            deltas = times.diff().dt.total_seconds()
            avg_interval = deltas.mean()
            return avg_interval
        except Exception:
            return np.nan

    return np.nan

def get_column_mean(df, keywords, exclude_keywords=None):
    if exclude_keywords is None:
        exclude_keywords = []

    df.columns = df.columns.str.strip().str.lower()
    for col in df.columns:
        if any(k in col for k in keywords) and not any(ek in col for ek in exclude_keywords):
            return pd.to_numeric(df[col], errors='coerce').mean()
    return np.nan

results = []

print(f"{'File':<10} | {'SNR (dB)':<10} | {'PSNR (dB)':<10} | {'MSE':<10} | {'Time (s)':<10}")
print("-" * 65)

for i, file_path in enumerate(log_files, 1):
    try:
        if os.path.exists(file_path):

            df = pd.read_csv(file_path, quotechar='"', skipinitialspace=True, on_bad_lines='skip')
            # ------------------
            snr_avg = get_column_mean(df, ['snr'], exclude_keywords=['psnr'])

            psnr_avg = get_column_mean(df, ['psnr'])
            mse_avg = get_column_mean(df, ['mse', 'mean_squared_error'])
            time_avg = calculate_time_metric(df)

            results.append({
                'snr': snr_avg,
                'psnr': psnr_avg,
                'mse': mse_avg,
                'time': time_avg
            })

            short_name = f"File {i}"
            t_disp = f"{time_avg:.4f}" if not np.isnan(time_avg) else "N/A"

            # Xử lý hiển thị nếu các chỉ số khác là NaN
            snr_disp = f"{snr_avg:.4f}" if not np.isnan(snr_avg) else "N/A"
            psnr_disp = f"{psnr_avg:.4f}" if not np.isnan(psnr_avg) else "N/A"
            mse_disp = f"{mse_avg:.6f}" if not np.isnan(mse_avg) else "N/A"

            print(f"{short_name:<10} | {snr_disp:<10} | {psnr_disp:<10} | {mse_disp:<10} | {t_disp:<10}")

        else:
            print(f"File {i:<5} | Lỗi: Không tìm thấy file")
            results.append({})

    except Exception as e:
        print(f"File {i:<5} | Lỗi: {str(e)}")
        results.append({})

print("-" * 65)

df_results = pd.DataFrame(results)
grand_avg = df_results.mean(numeric_only=True)

print(f"{'TB CHUNG':<10} | {grand_avg.get('snr', 0):<10.4f} | {grand_avg.get('psnr', 0):<10.4f} | {grand_avg.get('mse', 0):<10.6f} | {grand_avg.get('time', 0):<10.4f}")

File       | SNR (dB)   | PSNR (dB)  | MSE        | Time (s)  
-----------------------------------------------------------------
File 1     | 82.9372    | 106.7714   | 0.032541   | 23.5365   
File 2     | 82.9169    | 106.7731   | 0.032259   | 1.6953    
File 3     | 82.9397    | 106.7722   | 0.032473   | 1.9634    
File 4     | 82.9377    | 106.7718   | 0.032537   | 1.6784    
File 5     | 82.9370    | 106.7714   | 0.032543   | 1.7035    
File 6     | 17.8161    | 40.1532    | 0.001926   | 3.0725    
-----------------------------------------------------------------
TB CHUNG   | 72.0808    | 95.6688    | 0.027380   | 5.6083    


In [6]:
import pandas as pd
import os
import numpy as np

# Danh sách đường dẫn file
log_files = [
    "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/Timit_1000_0.065/logs/cnn_gen_timit_5_Random_Adaptive_ContentSalt_20260131_045203.csv",
    "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/Timit_1000_0.1/logs/cnn_gen_timit_5_Random_Adaptive_ContentSalt_20260130_145045.csv",
    "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/Timit_1000_0.2/logs/cnn_gen_timit_5_Random_Adaptive_ContentSalt_20260130_142941.csv",
    "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/Timit_1000_0.3/logs/cnn_gen_timit_5_Random_Adaptive_ContentSalt_20260130_150919.csv",
    "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/Timit_1000_0.4/logs/cnn_gen_timit_5_Random_Adaptive_ContentSalt_20260130_150005.csv",
    "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/Timit_1000_0.5/logs/cnn_gen_timit_5_Random_Adaptive_ContentSalt_20260131_043346.csv"
]

def calculate_time_metric(df):

    df.columns = df.columns.str.strip().str.lower()
    cols = df.columns.tolist()

    direct_time_cols = [c for c in cols if any(x in c for x in ['duration', 'elapsed', 'process']) and 'time' not in c]
    if not direct_time_cols:
         direct_time_cols = [c for c in cols if 'time' in c and 'stamp' not in c and 'date' not in c]

    if direct_time_cols:
        col_name = direct_time_cols[0]
        return pd.to_numeric(df[col_name], errors='coerce').mean()

    timestamp_cols = [c for c in cols if any(x in c for x in ['timestamp', 'date', 'created'])]

    if timestamp_cols:
        col_name = timestamp_cols[0]
        try:
            times = pd.to_datetime(df[col_name], errors='coerce')
            times = times.dropna().sort_values()

            if len(times) < 2: return np.nan

            deltas = times.diff().dt.total_seconds()
            avg_interval = deltas.mean()
            return avg_interval
        except Exception:
            return np.nan

    return np.nan

def get_column_mean(df, keywords, exclude_keywords=None):
    if exclude_keywords is None:
        exclude_keywords = []

    df.columns = df.columns.str.strip().str.lower()
    for col in df.columns:
        if any(k in col for k in keywords) and not any(ek in col for ek in exclude_keywords):
            return pd.to_numeric(df[col], errors='coerce').mean()
    return np.nan

results = []

print(f"{'File':<10} | {'SNR (dB)':<10} | {'PSNR (dB)':<10} | {'MSE':<10} | {'Time (s)':<10}")
print("-" * 65)

for i, file_path in enumerate(log_files, 1):
    try:
        if os.path.exists(file_path):

            df = pd.read_csv(file_path, quotechar='"', skipinitialspace=True, on_bad_lines='skip')
            # -------------------


            snr_avg = get_column_mean(df, ['snr'], exclude_keywords=['psnr'])

            psnr_avg = get_column_mean(df, ['psnr'])
            mse_avg = get_column_mean(df, ['mse', 'mean_squared_error'])

            # Tính thời gian
            time_avg = calculate_time_metric(df)

            results.append({
                'snr': snr_avg,
                'psnr': psnr_avg,
                'mse': mse_avg,
                'time': time_avg
            })

            short_name = f"File {i}"
            t_disp = f"{time_avg:.4f}" if not np.isnan(time_avg) else "N/A"

            # Xử lý hiển thị nếu các chỉ số khác là NaN
            snr_disp = f"{snr_avg:.4f}" if not np.isnan(snr_avg) else "N/A"
            psnr_disp = f"{psnr_avg:.4f}" if not np.isnan(psnr_avg) else "N/A"
            mse_disp = f"{mse_avg:.6f}" if not np.isnan(mse_avg) else "N/A"

            print(f"{short_name:<10} | {snr_disp:<10} | {psnr_disp:<10} | {mse_disp:<10} | {t_disp:<10}")

        else:
            print(f"File {i:<5} | Lỗi: Không tìm thấy file")
            results.append({})

    except Exception as e:
        print(f"File {i:<5} | Lỗi: {str(e)}")
        results.append({})

print("-" * 65)

df_results = pd.DataFrame(results)
grand_avg = df_results.mean(numeric_only=True)

print(f"{'TB CHUNG':<10} | {grand_avg.get('snr', 0):<10.4f} | {grand_avg.get('psnr', 0):<10.4f} | {grand_avg.get('mse', 0):<10.6f} | {grand_avg.get('time', 0):<10.4f}")

File       | SNR (dB)   | PSNR (dB)  | MSE        | Time (s)  
-----------------------------------------------------------------
File 1     | 67.8796    | 105.3228   | 0.031533   | 0.5181    
File 2     | 66.0349    | 103.4783   | 0.048216   | 0.5131    
File 3     | 63.0462    | 100.4899   | 0.095950   | 0.7975    
File 4     | 61.2945    | 98.7381    | 0.143611   | 0.5057    
File 5     | 60.0505    | 97.4940    | 0.191247   | 0.5079    
File 6     | 59.0825    | 96.5260    | 0.238991   | 0.5102    
-----------------------------------------------------------------
TB CHUNG   | 62.8980    | 100.3415   | 0.124925   | 0.5588    
